In [0]:
# Task 1: Incremental Azure SQL Extraction (Production Serverless Task)

from pyspark.sql import functions as F

# 1. Credentials from Secret Scope
JDBC_URL = dbutils.secrets.get(scope="banking_scope", key="azure_sql_jdbc_url")
JDBC_USER = dbutils.secrets.get(scope="banking_scope", key="azure_sql_user")
JDBC_PWD = dbutils.secrets.get(scope="banking_scope", key="azure_sql_password")
JDBC_DRIVER = "com.microsoft.sqlserver.jdbc.SQLServerDriver"

# 2. Incremental Extraction for Customers
def get_customer_watermark():
    try:
        row = spark.sql("SELECT MAX(last_updated) AS max_val FROM banking_lakehouse.bronze.customers_raw").first()
        if row and row["max_val"] is not None:
            return str(row["max_val"])
    except Exception:
        pass
    return "1970-01-01 00:00:00"

cust_watermark = get_customer_watermark()
operator = ">=" if cust_watermark == "1970-01-01 00:00:00" else ">"
print(f"Customer Watermark: {cust_watermark} (Filter: last_updated {operator} '{cust_watermark}')")

cust_query = f"""
(
    SELECT 
        customer_id, first_name, last_name, email, phone,
        address_street, city, state, postal_code, country,
        customer_segment, risk_score, kyc_status,
        created_at, last_updated, is_deleted
    FROM dbo.customers
    WHERE last_updated {operator} '{cust_watermark}'
) AS src_customers
"""

df_cust_delta = (
    spark.read.format("jdbc")
    .option("url", JDBC_URL)
    .option("driver", JDBC_DRIVER)
    .option("dbtable", cust_query)
    .option("user", JDBC_USER)
    .option("password", JDBC_PWD)
    .load()
    .withColumn("_ingest_timestamp", F.current_timestamp())
)

cust_count = df_cust_delta.count()
print(f"Extracted {cust_count} customer delta rows from Azure SQL.")

if cust_count > 0:
    df_cust_delta.write.format("delta").mode("append").saveAsTable("banking_lakehouse.bronze.customers_raw")
    print("Successfully appended customer changes to bronze.customers_raw.")

# 3. Incremental Extraction for Accounts
def get_account_watermark():
    try:
        row = spark.sql("SELECT MAX(last_updated) AS max_val FROM banking_lakehouse.bronze.accounts_raw").first()
        if row and row["max_val"] is not None:
            return str(row["max_val"])
    except Exception:
        pass
    return "1970-01-01 00:00:00"

acc_watermark = get_account_watermark()
acc_operator = ">=" if acc_watermark == "1970-01-01 00:00:00" else ">"
print(f"Account Watermark: {acc_watermark} (Filter: last_updated {acc_operator} '{acc_watermark}')")

acc_query = f"""
(
    SELECT 
        account_id, customer_id, account_type, currency,
        status, opened_at, last_updated, is_deleted
    FROM dbo.accounts
    WHERE last_updated {acc_operator} '{acc_watermark}'
) AS src_accounts
"""

df_acc_delta = (
    spark.read.format("jdbc")
    .option("url", JDBC_URL)
    .option("driver", JDBC_DRIVER)
    .option("dbtable", acc_query)
    .option("user", JDBC_USER)
    .option("password", JDBC_PWD)
    .load()
    .withColumn("_ingest_timestamp", F.current_timestamp())
)

acc_count = df_acc_delta.count()
print(f"Extracted {acc_count} account delta rows from Azure SQL.")

if acc_count > 0:
    df_acc_delta.write.format("delta").mode("append").saveAsTable("banking_lakehouse.bronze.accounts_raw")
    print("Successfully appended account changes to bronze.accounts_raw.")